# TransformerLens Setup & Hidden State Extraction

This notebook sets up TransformerLens with GPT-2 Small, runs a forward pass,
and extracts hidden states per layer per token.

**Model:** GPT-2 Small (124M params) — MacBook-friendly

**Goal:** Verify the inference pipeline works and we can access internal activations.

In [1]:
import torch
import numpy as np
from transformer_lens import HookedTransformer

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'mps' if torch.backends.mps.is_available() else 'cpu'}")
print(f"NumPy: {np.__version__}")

PyTorch: 2.2.2
Device: mps
NumPy: 1.26.4


## 1. Load GPT-2 Small via TransformerLens

In [2]:
# Load GPT-2 Small — TransformerLens wraps HuggingFace models
# and provides hooks at every layer/component
model = HookedTransformer.from_pretrained("gpt2-small")

print(f"Model: {model.cfg.model_name}")
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"d_model: {model.cfg.d_model}")
print(f"d_head: {model.cfg.d_head}")
print(f"Vocab size: {model.cfg.d_vocab}")
print(f"Context window: {model.cfg.n_ctx}")

Loaded pretrained model gpt2-small into HookedTransformer
Model: gpt2
Layers: 12
Heads: 12
d_model: 768
d_head: 64
Vocab size: 50257
Context window: 1024


## 2. Forward Pass with Cache

TransformerLens's `run_with_cache()` captures activations at every hook point.
This is the key primitive for mechanistic interpretability.

In [3]:
# Sample text for the forward pass
sample_text = "The capital of France is"

# Tokenize to see what we're working with
tokens = model.to_tokens(sample_text)
str_tokens = model.to_str_tokens(sample_text)

print(f"Input: '{sample_text}'")
print(f"Tokens shape: {tokens.shape}")
print(f"Token IDs: {tokens[0].tolist()}")
print(f"String tokens: {str_tokens}")

Input: 'The capital of France is'
Tokens shape: torch.Size([1, 6])
Token IDs: [50256, 464, 3139, 286, 4881, 318]
String tokens: ['<|endoftext|>', 'The', ' capital', ' of', ' France', ' is']


In [4]:
# Run forward pass and cache ALL activations
logits, cache = model.run_with_cache(sample_text)

print(f"Logits shape: {logits.shape}")
print(f"  → (batch={logits.shape[0]}, seq_len={logits.shape[1]}, vocab={logits.shape[2]})")
print(f"\nCache keys ({len(cache)} total):")
for i, key in enumerate(sorted(cache.keys())):
    if i < 15:
        print(f"  {key}: {cache[key].shape}")
print(f"  ... and {len(cache) - 15} more" if len(cache) > 15 else "")

Logits shape: torch.Size([1, 6, 50257])
  → (batch=1, seq_len=6, vocab=50257)

Cache keys (208 total):
  blocks.0.attn.hook_attn_scores: torch.Size([1, 12, 6, 6])
  blocks.0.attn.hook_k: torch.Size([1, 6, 12, 64])
  blocks.0.attn.hook_pattern: torch.Size([1, 12, 6, 6])
  blocks.0.attn.hook_q: torch.Size([1, 6, 12, 64])
  blocks.0.attn.hook_v: torch.Size([1, 6, 12, 64])
  blocks.0.attn.hook_z: torch.Size([1, 6, 12, 64])
  blocks.0.hook_attn_out: torch.Size([1, 6, 768])
  blocks.0.hook_mlp_out: torch.Size([1, 6, 768])
  blocks.0.hook_resid_mid: torch.Size([1, 6, 768])
  blocks.0.hook_resid_post: torch.Size([1, 6, 768])
  blocks.0.hook_resid_pre: torch.Size([1, 6, 768])
  blocks.0.ln1.hook_normalized: torch.Size([1, 6, 768])
  blocks.0.ln1.hook_scale: torch.Size([1, 6, 1])
  blocks.0.ln2.hook_normalized: torch.Size([1, 6, 768])
  blocks.0.ln2.hook_scale: torch.Size([1, 6, 1])
  ... and 193 more


## 3. Extract Hidden States (Residual Stream) Per Layer Per Token

The **residual stream** is the main information highway in transformers.
At each layer, attention and MLP outputs are added to it.

TransformerLens hook names:
- `blocks.{layer}.hook_resid_pre` — residual stream before the layer
- `blocks.{layer}.hook_resid_post` — residual stream after the layer
- `blocks.{layer}.hook_resid_mid` — after attention, before MLP

In [5]:
n_layers = model.cfg.n_layers
n_tokens = tokens.shape[1]

print(f"Extracting hidden states: {n_layers} layers × {n_tokens} tokens")
print(f"Each hidden state vector: d_model = {model.cfg.d_model}")
print()

# Collect residual stream post each layer
hidden_states = []
for layer in range(n_layers):
    key = f"blocks.{layer}.hook_resid_post"
    h = cache[key]  # shape: (batch, seq_len, d_model)
    hidden_states.append(h)
    
# Stack into (n_layers, batch, seq_len, d_model)
hidden_states = torch.stack(hidden_states)
print(f"Stacked hidden states shape: {hidden_states.shape}")
print(f"  → (layers={hidden_states.shape[0]}, batch={hidden_states.shape[1]}, "
      f"seq_len={hidden_states.shape[2]}, d_model={hidden_states.shape[3]})")

Extracting hidden states: 12 layers × 6 tokens
Each hidden state vector: d_model = 768

Stacked hidden states shape: torch.Size([12, 1, 6, 768])
  → (layers=12, batch=1, seq_len=6, d_model=768)


In [6]:
# Show per-layer, per-token norms (useful sanity check)
print("Residual stream L2 norms per layer per token:")
print(f"{'Layer':<8}", end="")
for tok in str_tokens:
    print(f"{repr(tok):<15}", end="")
print()
print("-" * (8 + 15 * n_tokens))

for layer in range(n_layers):
    norms = hidden_states[layer, 0].norm(dim=-1)  # (seq_len,)
    print(f"{layer:<8}", end="")
    for t in range(n_tokens):
        print(f"{norms[t].item():<15.2f}", end="")
    print()

Residual stream L2 norms per layer per token:
Layer   '<|endoftext|>''The'          ' capital'     ' of'          ' France'      ' is'          
--------------------------------------------------------------------------------------------------


0       165.21         57.04          59.54          57.15          66.68          59.21          
1       641.81         58.18          60.36          54.44          67.37          57.63          
2       2577.05        61.43          69.07          56.31          73.14          59.44          
3       2775.85        62.40          75.61          64.38          75.24          63.93          
4       2929.69        69.65          80.50          70.76          83.11          71.64          
5       3026.10        75.40          86.27          77.54          91.66          71.46          
6       3084.06        86.61          97.32          88.64          106.54         83.78          
7       3119.09        105.53         113.93         101.04         122.81         98.86          
8       3141.12        131.12         134.18         122.06         134.92         128.89         
9       3152.75        173.77         157.26         151.77         149.34         171.19         
10      31

## 4. Verify: Top Predictions at Final Token

Let's check the model's actual predictions for the last token position
("The capital of France is" → should predict "Paris").

In [7]:
# Get logits at the final token position
final_logits = logits[0, -1]  # (vocab_size,)
probs = torch.softmax(final_logits, dim=-1)

# Top 10 predictions
top_k = 10
top_probs, top_indices = probs.topk(top_k)

print(f"Top {top_k} predictions after '{sample_text}':")
print(f"{'Rank':<6} {'Token':<20} {'Prob':<10}")
print("-" * 36)
for i in range(top_k):
    token_str = model.to_string(top_indices[i].item())
    print(f"{i+1:<6} {repr(token_str):<20} {top_probs[i].item():.4f}")

Top 10 predictions after 'The capital of France is':
Rank   Token                Prob      
------------------------------------
1      ' now'               0.0475
2      ' the'               0.0374
3      ' a'                 0.0355
4      ' home'              0.0309
5      ' in'                0.0270
6      ' under'             0.0257
7      ' being'             0.0209
8      ' set'               0.0180
9      ' on'                0.0168
10     ' not'               0.0149


## 5. Bonus: Attention Patterns

Quick look at attention patterns — this is what we'll visualize later.

In [8]:
# Attention patterns from layer 0, all heads
attn_pattern = cache["blocks.0.attn.hook_pattern"]  # (batch, n_heads, seq_len, seq_len)
print(f"Attention pattern shape: {attn_pattern.shape}")
print(f"  → (batch={attn_pattern.shape[0]}, heads={attn_pattern.shape[1]}, "
      f"query_pos={attn_pattern.shape[2]}, key_pos={attn_pattern.shape[3]})")

# Show attention from final token to all tokens, head 0
print(f"\nLayer 0, Head 0 — attention FROM final token TO each token:")
for i, tok in enumerate(str_tokens):
    attn_val = attn_pattern[0, 0, -1, i].item()
    bar = "█" * int(attn_val * 50)
    print(f"  {repr(tok):<15} {attn_val:.4f} {bar}")

Attention pattern shape: torch.Size([1, 12, 6, 6])
  → (batch=1, heads=12, query_pos=6, key_pos=6)

Layer 0, Head 0 — attention FROM final token TO each token:
  '<|endoftext|>' 0.6630 █████████████████████████████████
  'The'           0.1566 ███████
  ' capital'      0.0682 ███
  ' of'           0.0259 █
  ' France'       0.0470 ██
  ' is'           0.0393 █


## Summary

✅ TransformerLens installed and working  
✅ GPT-2 Small loaded successfully  
✅ Forward pass with full activation cache  
✅ Hidden states extracted: shape `(n_layers, batch, seq_len, d_model)`  
✅ Predictions verified ("Paris" should be top prediction)  
✅ Attention patterns accessible  

### Key TransformerLens Concepts:
- `HookedTransformer.from_pretrained()` — loads model with hooks
- `model.run_with_cache()` — forward pass + capture all activations
- Cache keys follow pattern: `blocks.{layer}.{component}.hook_{name}`
- Residual stream: `hook_resid_pre`, `hook_resid_mid`, `hook_resid_post`
- Attention: `attn.hook_pattern`, `attn.hook_result`
- MLP: `mlp.hook_pre`, `mlp.hook_post`

### Next Steps:
1. Implement logit lens visualization
2. Build attention pattern visualizer
3. Explore specific circuits (e.g., induction heads)